# Test prior-guided refined pipeline (smoke notebook)

This notebook is a **small, interactive smoke test** for:

- `src/model/run_prior_guided_refined_pipeline.py`
- `utls/runners_prior.py::run_model_core_prior` item/trial outputs

It is intentionally lightweight so you can validate wiring before launching the full SLURM job.

## 1) Setup & imports

- Mocks `cox` (as in the pipeline script)
- Adds repo root to `sys.path`
- Imports the new refinement/transfer helpers

In [ ]:
import os
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Mock cox so constants import paths do not fail in notebook contexts.
cox_mock = types.ModuleType('cox')
store_mock = types.ModuleType('cox.store')
store_mock.PYTORCH_STATE = 'pytorch_state'
cox_mock.store = store_mock
sys.modules['cox'] = cox_mock
sys.modules['cox.store'] = store_mock

REPO_ROOT = Path.cwd()
REPO_ROOT = REPO_ROOT.parent


print(REPO_ROOT)
# if not (REPO_ROOT / 'src').exists():
#     raise RuntimeError(f'Run this notebook from repo root. cwd={REPO_ROOT}')

sys.path.insert(0, str(REPO_ROOT))

from src.model.run_prior_guided_refined_pipeline import (
    DEFAULT_ISIS,
    dense_grid_from_top_k,
    load_encoder_and_score,
    run_dense_refinement,
    run_single_isi_transfer,
)
from utls.runners_prior import run_model_core_prior
from utls.runners_utils import compute_human_curve, encode_stimuli, load_experiment_data
from utls.toy_experiments import make_high_diversity_sequences

print('Repo root:', REPO_ROOT)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set

## 2) Configure a tiny smoke run

In [9]:
# Update these paths to match your environment if needed.
COARSE_RESULTS = REPO_ROOT / 'reports' / 'figures' / 'prior_guided_grid_search_metric-euclidean_nmc1_task2' / 'grid_search_results_prior_guided.npz'
SMOKE_OUT = REPO_ROOT / 'reports' / 'figures' / 'prior_guided_refined_pipeline_smoke'
SMOKE_OUT.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Coarse results exists:', COARSE_RESULTS.exists(), COARSE_RESULTS)
print('Smoke output dir:', SMOKE_OUT)
print('Device:', DEVICE)

Coarse results exists: True /orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/reports/figures/prior_guided_grid_search_metric-euclidean_nmc1_task2/grid_search_results_prior_guided.npz
Smoke output dir: /orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/reports/figures/prior_guided_refined_pipeline_smoke
Device: cuda


## 3) Load multi-ISI data + shared model stack

In [19]:
which_task = 2  # auditory textures
isis = list(DEFAULT_ISIS)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(
    which_task=which_task,
    which_isi=None,
    is_multi=True,
)
print('Task:', task_name, '|', hr_task_name)
print('n stimuli:', len(all_files), '| n human runs:', len(human_runs))

# class _Args:
#     pc_dims = 256
#     device = DEVICE
#     score_config = '/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/bryan.yaml'
#     score_normalize = True


pc_dims = 256
encoder_cfg = dict(
    encoder_type='texture_pca',
    model_name='texture_pca',
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    device=DEVICE,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

pc_texture_model = AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params,
    sr=20000,
    rms_level=0.01,
    duration=2.0,
    device=device
)

print(f'Building prior encoder...')

encoder = pc_texture_model


print(f'Encoding {len(all_files)} stimuli ...')
X0 = encode_stimuli(encoder, all_files)
print(f'X0 shape: {X0.shape}  (D={X0.shape[1]})')

# ── setup: load learned score function ────────────────────────────
device = 'cuda'
score_model = ScoreFunction(mode='textures', restart=True, device=DEVICE)

# Quick sanity check on a few encoded stimuli
idx = torch.randperm(X0.shape[0], device=X0.device)[:8]
x_test = X0[idx]
with torch.no_grad():
    s_test = score_model.forward(x_test)

# ScoreFunction typically returns [B,1,1,D] for [B,D] input
if s_test.dim() == 4:
    s_flat = s_test.reshape(s_test.shape[0], -1)
elif s_test.dim() == 2:
    s_flat = s_test
else:
    raise ValueError(f'Unexpected score output shape: {tuple(s_test.shape)}')

norms = torch.norm(s_flat, dim=1)
print(f'score output shape: {tuple(s_test.shape)}')
print(f'score norms (first 8): {norms.detach().cpu().numpy()}')
print(f'All finite: {torch.isfinite(s_flat).all().item()}')
print(f'Any non-zero norm: {(norms > 0).any().item()}')


print('X0 shape:', tuple(X0.shape))
print('Finite check:', bool(torch.isfinite(X0).all()))

Task: atexts | Auditory Textures
n stimuli: 80 | n human runs: 120


NameError: name 'AudioTextureEncoderPCA' is not defined

## 4) Quick single-run check (item/trial outputs)

In [ ]:
stimulus_pool = sorted({s for seq in exp_list for s in seq})
smoke_sequences, _ = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=isis,
    n_sequences=4,
    length=50,
    min_pairs_per_isi=2,
    seed=44,
)

run = run_model_core_prior(
    sigma0=5.0,
    sigma=1.0,
    X0=X0,
    name_to_idx=name_to_idx,
    experiment_list=smoke_sequences,
    score_model=score_model,
    drift_step_size=0.1,
    metric='cosine',
    seed=44,
    return_item_scores=True,
    return_trial_log=True,
)

print('n hits:', len(run['hits']))
print('n fas:', len(run['fas']))
print('n item_hits keys:', len(run.get('item_hits', {})))
print('n item_fas keys:', len(run.get('item_fas', {})))
print('n trial log rows:', len(run.get('trial_log', [])))

assert 'item_hits' in run and 'item_fas' in run and 'trial_log' in run
assert np.isfinite(np.asarray(run['fas'], float)).all()

## 5) Tiny dense-range sanity check from coarse NPZ

In [ ]:
if not COARSE_RESULTS.exists():
    raise FileNotFoundError(
        f'Need coarse merged NPZ at {COARSE_RESULTS}. '
        'Set COARSE_RESULTS above to your merged grid-search artifact.'
    )

coarse = np.load(COARSE_RESULTS)
human_curve = compute_human_curve(human_runs, is_multi=True, which_isi=None)
info = dense_grid_from_top_k(
    coarse_npz=coarse,
    human_curve=human_curve,
    isis=np.array(isis, dtype=int),
    top_k=15,
    dense_points=5,
)

print('top sigma0 range:', float(np.min(info['top_sigma0'])), '->', float(np.max(info['top_sigma0'])))
print('top sigma  range:', float(np.min(info['top_sigma'])), '->', float(np.max(info['top_sigma'])))
print('top eta    range:', float(np.min(info['top_eta'])), '->', float(np.max(info['top_eta'])))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(info['dense_eta'], marker='o')
ax.set_title('Dense eta values (sanity)')
ax.set_xlabel('index')
ax.set_ylabel('eta')
ax.grid(alpha=0.2)
plt.show()

## 6) Run *tiny* end-to-end refinement + single-ISI transfer

In [ ]:
# Keep tiny for smoke test speed; scale up in SLURM script.
class SmokeArgs:
    coarse_results = str(COARSE_RESULTS)
    save_dir = str(SMOKE_OUT)

    which_task = 2
    isis = isis
    n_sequences = 4
    seq_length = 50
    min_pairs_per_isi = 2

    top_k = 5
    dense_points = 2
    n_mc = 1

    cv_splits = 2
    cv_frac = 0.6

    single_isi = 16
    single_isi_tasks = [0, 1, 2, 3]
    n_best_transfer = 2

    pc_dims = 256
    device = DEVICE
    metric = 'cosine'
    score_config = '/om2/user/bjmedina/auditory-memory/memory/assets/bryan.yaml'
    score_normalize = True
    seed = 44

smoke_args = SmokeArgs()
experiment_list = smoke_sequences

dense_df = run_dense_refinement(
    smoke_args,
    X0=X0,
    name_to_idx=name_to_idx,
    experiment_list=experiment_list,
    score_model=score_model,
    human_runs=human_runs,
)
print('Dense DF shape:', dense_df.shape)

top_df = dense_df.head(smoke_args.n_best_transfer).copy()
run_single_isi_transfer(smoke_args, encoder=encoder, score_model=score_model, best_df=top_df)

summary_csv = SMOKE_OUT / f'single_isi_{smoke_args.single_isi}' / 'single_isi_transfer_summary.csv'
print('Summary exists:', summary_csv.exists(), summary_csv)
if summary_csv.exists():
    display(pd.read_csv(summary_csv).head())

## 7) What to verify before scaling up

- `dense_refined_results.csv` was saved and has non-empty rows.
- `best_models_multi_isi.json` was saved.
- `single_isi_16/single_isi_transfer_summary.csv` exists.
- Per-model outputs exist for each task:
  - `*_trial_scores.csv`
  - `*_item_scores.json`
  - `*_run_summary.npz`

If this smoke notebook succeeds, you can run the full script with larger `dense_points`, `n_mc`, and `n_sequences` on SLURM.